# Check `run_vbr` results — `ecart_moyen` method

External sanity check of the pipeline output: starting from the raw quantitative data, we independently reproduce the center selection and compare it to the pipeline result.

The result file was produced with `qtrisk___ecart_moyen`, `frq___quarter`, `obswin___6`, `minnb___1`, `pai___complet`, verification proportions `plow=0.1 / pmid=0.15 / phigh=1.0`, for period `2026Q1`.

**Method (`ecart_moyen`), as implemented in the pipeline:**
- Observation **window** = quarters from `months_before(2026Q1, 6)` to `months_before(2026Q1, 4)` = **`2025Q3`, `2025Q4`**.
- **Indicator** = mean of `ecart_dec_val` over all window rows of the center.
- **Risk**: `> seum (0.1)` → high; `> seub (0.05)` → moderate; `>= 0` → low.
- **Eligibility**: `uneligible` if the center has no window period with a non-null `val`.
- **Verification**: each center is verified with probability = the proportion of its risk category (`low`=0.1, `moderate`=0.15, `high`=1.0, `uneligible`=1.0).

In [1]:
import pandas as pd

RESULT = (
    "model___model_july-month___2026Q1-frq___quarter-obswin___6-minnb___1"
    "-cvrf___100000-pai___complet-plow___0.1-pmid___0.15-phigh___1.0"
    "-qtrisk___ecart_moyen-seum___0.1-seub___0.05-vglow___25000-vgmod___75000-prov___nat.csv"
)

WINDOW = ["2025Q3", "2025Q4"]
SEUM, SEUB = 0.1, 0.05
PROPORTIONS = {"low": 0.1, "moderate": 0.15, "high": 1.0, "uneligible": 1.0}

quant = pd.read_csv("model_july_cleaned.csv", dtype={"month": str, "quarter": str})
result = pd.read_csv(RESULT)
len(quant), len(result)

(261578, 6717)

## Reproduce the selection from the quantitative data

In [2]:
window = quant[quant["quarter"].isin(WINDOW)]

ecart_moyen = window.groupby("ou")["ecart_dec_val"].mean()
eligible_ous = set(window.loc[window["val"].notna(), "ou"].unique())


def categorize(ou):
    if ou not in eligible_ous:
        return "uneligible"
    e = ecart_moyen.get(ou)
    if pd.isna(e):
        return "uneligible"
    if e > SEUM:
        return "high"
    if e > SEUB:
        return "moderate"
    return "low"


check = result[["ID", "Risk category", "Is the center verified?", "Mean ecart - window"]].copy()
check["ecart_recomputed"] = check["ID"].map(ecart_moyen)
check["risk_recomputed"] = check["ID"].map(categorize)
check.head()

,ID,Risk category,Is the center verified?,Mean ecart - window,ecart_recomputed,risk_recomputed
0,r7qP8eHP4X1,high,True,0.137828,0.137828,high
1,deCl1Cwbcky,low,False,0.008049,0.008049,low
2,fLOZ7hpShGI,moderate,False,0.056095,0.056095,moderate
3,S8au8vOSnpC,high,True,0.209947,0.209947,high
4,h9bYDEeuuTA,high,True,0.115717,0.115717,high


## Check 1 — the ecart indicator matches the pipeline

For every center with window data, our recomputed mean of `ecart_dec_val` must match the pipeline's `Mean ecart - window`.

In [3]:
has_ecart = check.dropna(subset=["Mean ecart - window", "ecart_recomputed"])
diff = (has_ecart["Mean ecart - window"] - has_ecart["ecart_recomputed"]).abs()

print(f"Centers with window data : {len(has_ecart)}")
print(f"Max absolute difference  : {diff.max():.2e}")
print(f"Ecart indicator matches  : {(diff < 1e-9).all()}")

Centers with window data : 6423
Max absolute difference  : 2.22e-16
Ecart indicator matches  : True


## Check 2 — full selection (eligibility + risk) reproduced

The independent recomputation of the risk category must match the pipeline for every center.

In [4]:
match_rate = (check["risk_recomputed"] == check["Risk category"]).mean()
print(f"Risk category match rate: {match_rate:.1%}\n")
pd.crosstab(
    check["Risk category"],
    check["risk_recomputed"],
    rownames=["pipeline"],
    colnames=["recomputed"],
    margins=True,
)

Risk category match rate: 100.0%



recomputed,high,low,moderate,uneligible,All
pipeline,,,,,
high,4157,0,0,0,4157
low,0,1462,0,0,1462
moderate,0,0,804,0,804
uneligible,0,0,0,294,294
All,4157,1462,804,294,6717


## Check 3 — verified centers respect the risk proportions

Verification is a random draw, so we can't reproduce exactly which centers were picked, but the share verified per risk category must match the configured proportion (high and uneligible are always verified).

In [5]:
verif = check.groupby("Risk category")["Is the center verified?"].agg(["sum", "count"])
verif["verified_rate"] = verif["sum"] / verif["count"]
verif["expected_proportion"] = verif.index.map(PROPORTIONS)
verif

,sum,count,verified_rate,expected_proportion
Risk category,,,,
high,4157,4157,1.000000,1.00
low,137,1462,0.093707,0.10
moderate,129,804,0.160448,0.15
uneligible,294,294,1.000000,1.00


## Conclusion

- **Check 1 passes**: the `ecart_moyen` indicator recomputed from the raw quantitative data matches the pipeline exactly.
- **Check 2 passes**: eligibility and risk categorization are reproduced for 100% of centers.
- **Check 3 passes**: high and uneligible centers are always verified, and the low / moderate verified shares match the configured `0.10` / `0.15` proportions (up to random sampling).

The pipeline's center selection under the `ecart_moyen` method is externally consistent with the quantitative data.

# Check `run_vbr` results — `verifgain` method

Same idea, second result file: starting from the raw quantitative data we independently reproduce the verification-gain indicator and the center selection, and compare to the pipeline.

This file was produced with `qtrisk___verifgain`, `frq___quarter`, `obswin___6`, `minnb___1`, **`pai___tauxval`**, verification proportions `plow=0.1 / pmid=0.5 / phigh=1.0`, `vglow=25000 / vgmod=75000`, for period `2026Q1`.

**Method (`verifgain`), as implemented in the pipeline (`calculate_gain_verification_window` + `categorize_quantity_verifgain`):**
- Observation **window** = `2025Q3`, `2025Q4` (same as before).
- For **each window period**, the gain of verifying is
  `cout + Σ_service ( subside_avec_verification − subside_sans_verification × factor )`,
  where, under `tauxval`, `factor` is the **per-service mean of `taux_validation`** over the *two quarters preceding that window period* (`get_previous_two_periods`), falling back to the median-over-services (service absent) or to the window quarters (no preceding data). The indicator `benefice_vbr_window` is the **mean of the per-period gains** — the pipeline stores it in the `Median benefice complet VBR - window` column.
- **Risk** (`categorize_quantity_verifgain`): `benefice > vglow (25000)` → low; `elif benefice > vgmod (75000)` → moderate; `else` → high.
- **Eligibility / verification**: same as the `ecart` method.

> ⚠️ **Pipeline quirk we expect to see, not a bug in this check.** Because `vglow (25000) < vgmod (75000)`, any benefice above `75000` already satisfies the first `if` (`> 25000`) and is labelled **low** — so the `elif ... > 75000 → moderate` branch is **unreachable** and **no center is ever categorised `moderate`** under `verifgain`. The recomputation below replicates the pipeline's exact `if/elif/else`, so it reproduces this behaviour (0 moderate) rather than "correcting" it.

In [6]:
RESULT_VG = (
    "model___model_july-month___2026Q1-frq___quarter-obswin___6-minnb___1"
    "-cvrf___100000-pai___tauxval-plow___0.1-pmid___0.5-phigh___1.0"
    "-qtrisk___verifgain-seum___0.1-seub___0.05-vglow___25000-vgmod___75000-prov___nation.csv"
)

COUT = 100000
VG_LOW, VG_MOD = 25000, 75000
PROPORTIONS_VG = {"low": 0.1, "moderate": 0.5, "high": 1.0, "uneligible": 1.0}

result_vg = pd.read_csv(RESULT_VG)
len(quant), len(result_vg)

(261578, 6717)

## Reproduce the verification-gain indicator from the quantitative data

We rebuild `benefice_vbr_window` per center with the same `tauxval` factor logic as the pipeline, fully vectorised (no per-center Python loop, so it runs in a few seconds).

In [7]:
import numpy as np


def quarter_months_before(q, lag):
    """Quarter `lag` months before `q` (YYYYQX), mirroring orgunit.months_before."""
    year, quart = q.split("Q")
    total = int(year) * 12 + int(quart) * 3 - lag
    ny, nm = total // 12, total % 12
    if nm == 0:
        ny -= 1
        nm = 12
    return f"{ny}Q{(nm - 1) // 3 + 1}"


# For tauxval, the factor of a (center, window_period, service) is the mean of taux_validation
# over the center's rows in the two quarters preceding that window period.
ref_quarters = {p: [quarter_months_before(p, 3), quarter_months_before(p, 6)] for p in WINDOW}

win_rows = quant[quant["quarter"].isin(WINDOW)]

# per-service mean taux over the reference quarters, per (center, window_period)
fac = []
for period in WINDOW:
    ref = quant[quant["quarter"].isin(ref_quarters[period])]
    sm = (
        ref.groupby(["ou", "service"], observed=True)["taux_validation"]
        .mean()
        .reset_index(name="factor")
    )
    sm["window_period"] = period
    fac.append(sm)
fac = pd.concat(fac, ignore_index=True)
med_factor = (
    fac.groupby(["ou", "window_period"], observed=True)["factor"]
    .median()
    .reset_index(name="med_factor")
)
n_ref = fac.groupby(["ou", "window_period"], observed=True).size().reset_index(name="n_ref")

# fallback used by the pipeline when a window period has no preceding data at all: the window quarters themselves
sm_win = (
    win_rows.groupby(["ou", "service"], observed=True)["taux_validation"]
    .mean()
    .reset_index(name="factor_win")
)
med_win = sm_win.groupby("ou")["factor_win"].median().reset_index(name="med_win")

# window-period subsidies per (center, window_period, service)
w = (
    win_rows.groupby(["ou", "quarter", "service"], observed=True)
    .agg(ssv=("subside_sans_verification", "sum"), sav=("subside_avec_verification", "sum"))
    .reset_index()
    .rename(columns={"quarter": "window_period"})
)
w = (
    w.merge(fac, on=["ou", "service", "window_period"], how="left")
    .merge(med_factor, on=["ou", "window_period"], how="left")
    .merge(n_ref, on=["ou", "window_period"], how="left")
    .merge(sm_win, on=["ou", "service"], how="left")
    .merge(med_win, on="ou", how="left")
)
w["n_ref"] = w["n_ref"].fillna(0)

# resolve the factor: from reference quarters if any exist (median fallback for a missing service),
# otherwise from the window quarters.
factor_ref = w["factor"].where(w["factor"].notna(), w["med_factor"])
factor_win = w["factor_win"].where(w["factor_win"].notna(), w["med_win"])
w["factor"] = np.where(w["n_ref"] > 0, factor_ref, factor_win)

# gain per (center, window_period) = cout + Σ_service (sav − ssv × factor); indicator = mean over periods
w["gain_service"] = -w["ssv"] * w["factor"] + w["sav"]
gain_period = w.groupby(["ou", "window_period"], observed=True)["gain_service"].sum() + COUT
benefice_recomputed = gain_period.groupby("ou").mean()

benefice_recomputed.head()

ou
A16VrV00Wzq     63882.456504
A33hqse1xhG     93350.000000
A3lBmWhvL0B    161562.500000
A3ycPUINGnD     23875.000000
A4TRdvnqXVm    101711.120690
Name: gain_service, dtype: float64

In [8]:
# eligibility: same rule as the ecart check (minnb=1 -> at least one window quarter with a non-null val)
eligible_ous_vg = set(win_rows.loc[win_rows["val"].notna(), "ou"].unique())


def categorize_vg(ou):
    if ou not in eligible_ous_vg:
        return "uneligible"
    b = benefice_recomputed.get(ou)
    if pd.isna(b):
        return "uneligible"
    # Replicates categorize_quantity_verifgain exactly (note: the moderate branch is unreachable
    # because VG_LOW < VG_MOD -- see the warning above).
    if b > VG_LOW:
        return "low"
    elif b > VG_MOD:
        return "moderate"
    else:
        return "high"


check_vg = result_vg[
    ["ID", "Risk category", "Is the center verified?", "Median benefice complet VBR - window"]
].copy()
check_vg["benefice_recomputed"] = check_vg["ID"].map(benefice_recomputed)
check_vg["risk_recomputed"] = check_vg["ID"].map(categorize_vg)
check_vg.head()

,ID,Risk category,Is the center verified?,Median benefice complet VBR - window,benefice_recomputed,risk_recomputed
0,r7qP8eHP4X1,low,True,88886.569215,88886.569215,low
1,deCl1Cwbcky,low,False,110558.873450,110558.873450,low
2,fLOZ7hpShGI,low,False,231710.414880,231710.414880,low
3,S8au8vOSnpC,low,False,184316.137142,184316.137142,low
4,h9bYDEeuuTA,low,False,73437.730101,73437.730101,low


## Check 1 (verifgain) — the verification-gain indicator matches the pipeline

For every center with window data, our recomputed `benefice_vbr_window` must match the pipeline's `Median benefice complet VBR - window`.

In [9]:
has_ben = check_vg.dropna(subset=["Median benefice complet VBR - window", "benefice_recomputed"])
diff_vg = (has_ben["Median benefice complet VBR - window"] - has_ben["benefice_recomputed"]).abs()

print(f"Centers with window data      : {len(has_ben)}")
print(f"Max absolute difference       : {diff_vg.max():.2e}")
print(f"Verifgain indicator matches   : {(diff_vg < 1e-6).all()}")

Centers with window data      : 6423
Max absolute difference       : 1.51e-09
Verifgain indicator matches   : True


## Check 2 (verifgain) — full selection (eligibility + risk) reproduced

The independent recomputation of the risk category must match the pipeline for every center. We expect the `moderate` column to be empty (see the warning above).

In [10]:
match_rate_vg = (check_vg["risk_recomputed"] == check_vg["Risk category"]).mean()
print(f"Risk category match rate: {match_rate_vg:.1%}\n")
pd.crosstab(
    check_vg["Risk category"],
    check_vg["risk_recomputed"],
    rownames=["pipeline"],
    colnames=["recomputed"],
    margins=True,
)

Risk category match rate: 100.0%



recomputed,high,low,uneligible,All
pipeline,,,,
high,296,0,0,296
low,0,6127,0,6127
uneligible,0,0,294,294
All,296,6127,294,6717


## Check 3 (verifgain) — verified centers respect the risk proportions

As before, verification is a random draw, so we check the share verified per risk category against the configured proportion (`plow=0.1 / pmid=0.5 / phigh=1.0`; high and uneligible always verified). The `moderate` row is expected to be absent.

In [11]:
verif_vg = check_vg.groupby("Risk category")["Is the center verified?"].agg(["sum", "count"])
verif_vg["verified_rate"] = verif_vg["sum"] / verif_vg["count"]
verif_vg["expected_proportion"] = verif_vg.index.map(PROPORTIONS_VG)
verif_vg

,sum,count,verified_rate,expected_proportion
Risk category,,,,
high,296,296,1.000000,1.0
low,676,6127,0.110331,0.1
uneligible,294,294,1.000000,1.0


## Conclusion (verifgain)

- **Check 1 passes**: the `benefice_vbr_window` indicator recomputed from the raw data — including the full `tauxval` factor logic — matches the pipeline to floating-point precision.
- **Check 2 passes**: eligibility and risk categorisation are reproduced for 100% of centers. As expected, **no center is `moderate`** — this is the pipeline's unreachable-branch behaviour (`vglow < vgmod`), faithfully reproduced, not a discrepancy.
- **Check 3 passes**: high and uneligible centers are always verified, and the low verified share matches the configured `0.10` proportion (up to random sampling). There are no moderate centers to check against `0.5`.

The pipeline's center selection under the `verifgain` method is externally consistent with the quantitative data. The only thing to flag is the **product logic**: with `vglow (25000) < vgmod (75000)`, the `moderate` category can never be assigned — if moderate-risk centers are intended under `verifgain`, the thresholds in `categorize_quantity_verifgain` need revisiting.